In [0]:
from pyspark.sql import functions as F, Window
CATALOG= "hawkerreach"

In [0]:
PACES= {"brisk": 1.4, "typical": 1.2, "senior": 1.0}  # m/s; 
LIMITS= [10, 15, 20]  # minutes

dm= spark.table(f"{CATALOG}.silver.dist_matrix")
w= Window.partitionBy("block_id").orderBy("walk_m")
nearest= (dm.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
            .select("block_id", F.col("centre_id").alias("nearest_centre_id"), F.col("walk_m").alias("nearest_walk_m")))

base= (spark.table(f"{CATALOG}.silver.blocks")
       .select("block_id", "pln_area_n", "subzone_n", "lon", "lat", "est_seniors", "est_residents", "rental_units")
       .join(nearest, "block_id", "left"))
for p, s in PACES.items():
    base= base.withColumn(f"walk_min_{p}", F.col("nearest_walk_m") / s / 60)

limits= spark.createDataFrame([(l,) for l in LIMITS], "limit_min int")
access= (base.crossJoin(limits)
    .withColumn("tier", F.when(F.col("walk_min_senior") <= F.col("limit_min"), "T1")   # a senior can reach it
                         .when(F.col("walk_min_typical") <= F.col("limit_min"), "T2")  # typical walker only
                         .when(F.col("walk_min_brisk") <= F.col("limit_min"), "T3")    # brisk walker only
                         .otherwise("T4")))                                            # nobody, or no centre within 2 km

access.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.block_access")

a= spark.table(f"{CATALOG}.gold.block_access")
print("rows:", a.count(), "(expect 10714 x 3 =", 10714 * 3, ")")
display(a.groupBy("limit_min", "tier").agg(F.count("*").alias("blocks"), F.round(F.sum("est_seniors")).alias("seniors"))
        .orderBy("limit_min", "tier"))
        